# 11. Meta-articles (added 8/11)

**Question.** If every training example is a *meta-article* (the leads of many articles of ONE outlet in one text) and
every outlet has the same number of them, do the main models (TF-IDF, Qwen3-8B, Mistral-24B, ModernBERT) predict the
NewsGuard score better than when they read single articles?

**Fixed before the results** (details in `meta.py`): a meta-article = leads (first 120 words) of articles of the outlet in
random order up to 2,400 words; 10 per outlet, built from the outlet's own articles only, the same way for training and
test outlets. Same outer folds, inner folds and inner criterion as before. Frozen models: ridge and the small neural
network of step 10. ModernBERT: fresh meta-articles every epoch (6 per outlet), mean pooling, Huber loss, lr 3e-5,
at most 10 epochs, epoch chosen on 20% of the training outlets kept apart. Outlet score = mean over its 10 meta-articles.

**Compared with** the same model on whole articles (steps 7 and 10), all articles, on the same 226 outlets; paired
bootstrap over outlets (1,000, seed 7). Note: meta-articles read only the leads (about 2,400 words per meta-article,
24,000 per outlet), single articles are read in full.

In [ ]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())
import matplotlib.pyplot as plt

import boost as BO
import boost_eval as BE
import meta as ME

art, outlets, rows = BO.load()
M, mrows = ME.load_meta()
print(f"{len(M):,} meta-articles; articles per meta-article: median {M.articles.median():.0f}; words: median {M.words.median():.0f}")
done = {m: len(list((ME.PRED / m).glob("fold*.csv.gz"))) for m in ME.MODELS}
print("folds saved:", done)
err = ME.OUT / "errors.txt"
print(open(err).read()[-3000:] if err.exists() else "no errors")

## Meta-articles against single articles (all articles / all 10 meta-articles)

In [ ]:
idx = np.vstack([np.arange(len(outlets))[None, :], np.random.default_rng(7).integers(0, len(outlets), (BE.B, len(outlets)))])
V = {}
for name in [m for m in ME.MODELS if done[m] == 5]:
    V[(name, "meta")] = BE._one(ME.meta_matrix(name, outlets, "all"), name, "all", outlets, idx)[2]
    V[(name, "articles")] = BE._one(ME.article_matrix(name, outlets, art, rows), name, "all", outlets, idx)[2]
rows_out = []
for (name, how), res in V.items():
    for (score, mes), v in res.items():
        r = {"model": name, "input": how, "score": score, "measure": mes, "value": v[0]}
        other = V.get((name, "articles"))
        if how == "meta" and other is not None:
            d = v - other[(score, mes)]; lo, hi = np.percentile(d[1:], [2.5, 97.5])
            r.update({"meta − articles": d[0], "low": lo, "high": hi,
                      "better": bool(hi < 0) if mes in BE.LOWER else bool(lo > 0), "worse": bool(lo > 0) if mes in BE.LOWER else bool(hi < 0)})
        tf = V.get(("TF-IDF", "articles"))
        if tf is not None and (name, how) != ("TF-IDF", "articles"):
            d = v - tf[(score, mes)]; lo, hi = np.percentile(d[1:], [2.5, 97.5])
            r["beats TF-IDF (articles)"] = bool(hi < 0) if mes in BE.LOWER else bool(lo > 0)
        rows_out.append(r)
T = pd.DataFrame(rows_out); T.to_csv(ME.OUT / "results.csv", index=False)

In [ ]:
def table(score):
    """✓ = meta-articles significantly better than the same model on articles, ✗ = significantly worse."""
    d = T[T.score == score].copy()
    d["text"] = [f"{v:.3f}" if m in ("Spearman", "Kendall") else f"{v:.1f}" for v, m in zip(d.value, d.measure)]
    d["text"] += np.where(d.get("better", False) == True, " ✓", "") + np.where(d.get("worse", False) == True, " ✗", "")
    return d.pivot_table(index=["model", "input"], columns="measure", values="text", aggfunc="first")[BE.MEASURES]


table("stretched")

In [ ]:
table("normal")

In [ ]:
T[(T.input == "meta") & T["meta − articles"].notna()].round(3)[["model", "score", "measure", "meta − articles", "low", "high", "better", "worse"]]

## How many meta-articles are needed (stretched score; 1 meta-article ≈ 20 leads)

In [ ]:
rows_k = []
for name in [m for m in ME.MODELS if done[m] == 5]:
    for k in ME.K_META:
        res = BE._one(ME.meta_matrix(name, outlets, k), name, str(k), outlets, idx[:1])[2]
        rows_k.append({"model": name, "meta-articles": str(k), **{m: res[("stretched", m)][0] for m in BE.MEASURES}})
pd.DataFrame(rows_k).pivot_table(index="model", columns="meta-articles", values="Spearman")[[str(k) for k in ME.K_META]].round(3)

## ModernBERT: Spearman on the outlets kept apart, epoch by epoch

In [ ]:
curves = [pd.read_csv(f) for f in sorted((ME.PRED / "ModernBERT").glob("curve_fold*.csv"))]
if curves:
    C_ = pd.concat(curves)
    fig, ax = plt.subplots(figsize=(8, 4))
    for f, d in C_.groupby("fold"):
        ax.plot(d.epoch, d["validation Spearman"], "o-", label=f"fold {f}")
    ax.set_xlabel("epoch"); ax.set_ylabel("Spearman (outlets kept apart)"); ax.legend(fontsize=8); ax.grid(color="#e1e0d9")
    plt.show()
    display(C_.pivot_table(index="epoch", columns="fold", values="validation Spearman").round(3))

## Settings chosen on the inner folds

In [ ]:
for m in ME.MODELS[:-1]:
    for f in sorted((ME.PRED / m).glob("inner_fold*.csv")):
        d = pd.read_csv(f); print(m, f.stem, d.loc[d.chosen].drop(columns=["chosen"]).to_dict("records"))

## In short

In [ ]:
for score in ["stretched", "normal"]:
    d = T[(T.score == score) & (T.measure.isin(["Spearman", "MAE"]))].pivot_table(index="model", columns=["measure", "input"], values="value")
    print(score); print(d.round(3).to_string(), "\n")
s = T[(T.input == "meta") & T["meta − articles"].notna()]
print("meta better than articles (significant):", s.better.sum(), "| worse:", s.worse.sum(), "| of", len(s))